# DocLite: SROIE Experiments

This notebook runs all three experiments on the **SROIE** dataset (receipts):

1. **LayoutLMv3 Baseline** (teacher, supervised) — upper bound
2. **LiLT Baseline** (student, supervised) — lower bound
3. **LiLT + DocLite Distillation** (student with teacher knowledge) — our contribution

**Task:** Token classification on scanned receipts (company / date / address / total / other)

**Metrics:** Token-level accuracy and micro-F1

## Dataset Setup

Download SROIE from: https://github.com/zzzDavid/ICDAR-2019-SROIE

Place it so the directory structure is:
```
data/sroie/
  train/
    box/       <- OCR .txt files (x1,y1,...,x4,y4,text)
    entities/  <- Entity .txt files (JSON with company/date/address/total)
  test/
    box/
    entities/
```

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, LayoutLMv3Config, LayoutLMv3ForTokenClassification

from doclite.configs.core import ENV, WEIGHTS
from doclite.data.document_dataset import DocumentDataset
from doclite.data.collate import collate_document_batch
from doclite.eval.evaluate import evaluate_student
from doclite.utils.checkpoint import save_checkpoint
from doclite.models.teacher import TeacherModel
from doclite.models.student import StudentModel
from doclite.distill.distill_loss import compute_distill_loss
from build_sroie_examples import load_sroie_split, NUM_LABELS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load & Inspect SROIE Data

In [ ]:
SROIE_ROOT = ENV.DATA / "sroie"
TRAIN_OCR_DIR = SROIE_ROOT / "train" / "box"
TRAIN_ENT_DIR = SROIE_ROOT / "train" / "entities"
TEST_OCR_DIR = SROIE_ROOT / "test" / "box"
TEST_ENT_DIR = SROIE_ROOT / "test" / "entities"

tokenizer = AutoTokenizer.from_pretrained("microsoft/layoutlmv3-base")

train_examples = load_sroie_split(TRAIN_OCR_DIR, TRAIN_ENT_DIR, tokenizer)
test_examples = load_sroie_split(TEST_OCR_DIR, TEST_ENT_DIR, tokenizer)

print(f"Training receipts: {len(train_examples)}")
print(f"Testing receipts:  {len(test_examples)}")

In [ ]:
# Create DataLoaders
BATCH_SIZE = 2

train_dataset = DocumentDataset(train_examples)
test_dataset = DocumentDataset(test_examples)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_document_batch)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
# Inspect label distribution
from collections import Counter

ID2LABEL = {0: "company", 1: "date", 2: "address", 3: "total", 4: "other", -100: "[PAD]"}

all_labels = []
for ex in train_examples:
    all_labels.extend([l for l in ex["labels"] if l != -100])

label_counts = Counter(all_labels)
print("Label distribution across training set:")
for label_id, count in sorted(label_counts.items()):
    print(f"  {ID2LABEL.get(label_id, label_id):>10s}: {count} ({count/len(all_labels)*100:.1f}%)")

## 2. Experiment A: LayoutLMv3 Baseline (Teacher Upper Bound)

In [ ]:
NUM_EPOCHS = 3
LR = 1e-5

config = LayoutLMv3Config.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=NUM_LABELS,
    output_hidden_states=True,
    output_attentions=True,
)

teacher_model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", config=config
).to(device)

optimizer = torch.optim.AdamW(teacher_model.parameters(), lr=LR, weight_decay=0.01)

class LayoutLMv3EvalWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask, bbox):
        out = self.model(input_ids=input_ids, attention_mask=attention_mask, bbox=bbox)
        return {"logits": out.logits}

eval_wrapper = LayoutLMv3EvalWrapper(teacher_model).to(device)
print(f"LayoutLMv3 parameters: {sum(p.numel() for p in teacher_model.parameters()):,}")

In [ ]:
teacher_results = []

for epoch in range(NUM_EPOCHS):
    teacher_model.train()
    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        outputs = teacher_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            bbox=batch["bbox"],
            labels=batch["labels"],
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if step % 25 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | loss={loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    metrics = evaluate_student(eval_wrapper, test_loader, device=device)
    teacher_results.append(metrics)

    print(f"Epoch {epoch+1} | train_loss={avg_loss:.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\nLayoutLMv3 Best F1: {max(r['token_f1'] for r in teacher_results):.4f}")

## 3. Experiment B: LiLT Baseline (Student Lower Bound)

In [ ]:
student_baseline = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS).to(device)
optimizer_sb = torch.optim.AdamW(student_baseline.parameters(), lr=LR, weight_decay=0.01)

print(f"LiLT parameters: {sum(p.numel() for p in student_baseline.parameters()):,}")

In [ ]:
student_baseline_results = []

for epoch in range(NUM_EPOCHS):
    student_baseline.train()
    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_sb.zero_grad()

        outputs = student_baseline.model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            bbox=batch["bbox"],
            labels=batch["labels"],
        )

        loss = outputs.loss
        loss.backward()
        optimizer_sb.step()
        total_loss += loss.item()

        if step % 25 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | loss={loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    metrics = evaluate_student(student_baseline, test_loader, device=device)
    student_baseline_results.append(metrics)

    print(f"Epoch {epoch+1} | train_loss={avg_loss:.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\nLiLT Baseline Best F1: {max(r['token_f1'] for r in student_baseline_results):.4f}")

## 4. Experiment C: DocLite Distillation (Our Contribution)

In [ ]:
teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=NUM_LABELS).to(device)
for param in teacher.parameters():
    param.requires_grad = False

student_distill = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS).to(device)
optimizer_sd = torch.optim.AdamW(student_distill.parameters(), lr=LR, weight_decay=0.01)

print(f"Distillation weights: ALPHA={WEIGHTS.ALPHA_HIDDEN}, BETA={WEIGHTS.BETA_ATTN}, GAMMA={WEIGHTS.GAMMA_LOGITS}, DELTA={WEIGHTS.DELTA_TASK}")

In [ ]:
distill_results = []
best_f1 = -1.0

for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student_distill.train()

    total_loss = 0.0
    total_distill = 0.0
    total_task = 0.0

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_sd.zero_grad()

        model_inputs = {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"],
            "bbox": batch["bbox"],
        }

        teacher_out = teacher(**model_inputs)

        student_hf_out = student_distill.model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            bbox=batch["bbox"],
            labels=batch["labels"],
            output_hidden_states=True,
            output_attentions=True,
            return_dict=True,
        )

        student_out = {
            "logits": student_hf_out.logits,
            "hidden_states": student_hf_out.hidden_states,
            "attentions": student_hf_out.attentions,
        }

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        distill_loss = distill_losses["loss_total"]
        task_loss = student_hf_out.loss

        loss = distill_loss + WEIGHTS.DELTA_TASK * task_loss
        loss.backward()
        optimizer_sd.step()

        total_loss += loss.item()
        total_distill += distill_loss.item()
        total_task += task_loss.item()

        if step % 25 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | total={loss.item():.4f} | distill={distill_loss.item():.4f} | task={task_loss.item():.4f}")

    avg_total = total_loss / len(train_loader)
    metrics = evaluate_student(student_distill, test_loader, device=device)
    distill_results.append(metrics)

    if metrics["token_f1"] > best_f1:
        best_f1 = metrics["token_f1"]

    print(f"Epoch {epoch+1} | train_loss={avg_total:.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\nDocLite Best F1: {best_f1:.4f}")

## 5. Results Comparison

In [ ]:
print("=" * 65)
print(f"{'Model':<30s} {'Accuracy':>10s} {'F1':>10s} {'Params':>12s}")
print("=" * 65)

teacher_best = max(teacher_results, key=lambda x: x['token_f1'])
student_best = max(student_baseline_results, key=lambda x: x['token_f1'])
distill_best = max(distill_results, key=lambda x: x['token_f1'])

teacher_params = sum(p.numel() for p in teacher_model.parameters())
student_params = sum(p.numel() for p in student_distill.parameters())

print(f"{'LayoutLMv3 (teacher)':<30s} {teacher_best['token_acc']:>10.4f} {teacher_best['token_f1']:>10.4f} {teacher_params:>12,}")
print(f"{'LiLT (baseline)':<30s} {student_best['token_acc']:>10.4f} {student_best['token_f1']:>10.4f} {student_params:>12,}")
print(f"{'LiLT + DocLite (ours)':<30s} {distill_best['token_acc']:>10.4f} {distill_best['token_f1']:>10.4f} {student_params:>12,}")
print("=" * 65)

gap_closed = (distill_best['token_f1'] - student_best['token_f1']) / (teacher_best['token_f1'] - student_best['token_f1'] + 1e-8) * 100
print(f"\nDistillation closed {gap_closed:.1f}% of the teacher-student F1 gap")